In [27]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [28]:
df = pd.read_csv('../data/WineQT.csv')

In [29]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4


In [30]:
df.dropna(inplace=True)
df.drop(['Id'], axis=1, inplace=True)

In [31]:
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 107.3 KB


In [33]:
X_raw = df.drop('quality', axis=1).values
Y_raw = df['quality'].values

In [34]:
X_raw, Y_raw

(array([[ 7.4  ,  0.7  ,  0.   , ...,  3.51 ,  0.56 ,  9.4  ],
        [ 7.8  ,  0.88 ,  0.   , ...,  3.2  ,  0.68 ,  9.8  ],
        [ 7.8  ,  0.76 ,  0.04 , ...,  3.26 ,  0.65 ,  9.8  ],
        ...,
        [ 6.2  ,  0.6  ,  0.08 , ...,  3.45 ,  0.58 , 10.5  ],
        [ 5.9  ,  0.55 ,  0.1  , ...,  3.52 ,  0.76 , 11.2  ],
        [ 5.9  ,  0.645,  0.12 , ...,  3.57 ,  0.71 , 10.2  ]],
       shape=(1143, 11)),
 array([5, 5, 5, ..., 5, 6, 5], shape=(1143,)))

In [35]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_raw)

In [36]:
X_scaled, X_scaled.shape

(array([[-0.52157961,  0.93933222, -1.36502663, ...,  1.27069495,
         -0.57365783, -0.96338181],
        [-0.29259344,  1.94181282, -1.36502663, ..., -0.70892755,
          0.1308811 , -0.59360107],
        [-0.29259344,  1.27349242, -1.16156762, ..., -0.32577481,
         -0.04525363, -0.59360107],
        ...,
        [-1.20853813,  0.38239855, -0.9581086 , ...,  0.88754221,
         -0.45623467,  0.05351522],
        [-1.38027776,  0.10393172, -0.8563791 , ...,  1.33455374,
          0.60057372,  0.70063152],
        [-1.38027776,  0.6330187 , -0.75464959, ...,  1.65384769,
          0.30701583, -0.22382033]], shape=(1143, 11)),
 (1143, 11))

In [37]:
target_scaler = StandardScaler()

Y_scaled = target_scaler.fit_transform(Y_raw.reshape(-1, 1))

In [38]:
Y_scaled, Y_scaled.shape

(array([[-0.81572437],
        [-0.81572437],
        [-0.81572437],
        ...,
        [-0.81572437],
        [ 0.42578423],
        [-0.81572437]], shape=(1143, 1)),
 (1143, 1))

In [39]:
X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y_scaled, test_size=0.2, random_state=42)

In [42]:

from torch.utils.data import Dataset, DataLoader

class WineQualityDataset(Dataset):
    def __init__(self, features, targets):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.Y = torch.tensor(targets, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

In [44]:
train_dataset = WineQualityDataset(features=X_train, targets=Y_train)

test_dataset = WineQualityDataset(features=X_test, targets=Y_test)

In [45]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

for batch_X, batch_y in train_loader:
    print("Batch X Shape:", batch_X.shape)
    print("Batch y Shape:", batch_y.shape)
    break

Batch X Shape: torch.Size([32, 11])
Batch y Shape: torch.Size([32, 1])


In [46]:
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [47]:
class WineQualityModel(nn.Module):
    def __init__(self, input_dim):
        super(WineQualityModel, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 16)
        self.fc4 = nn.Linear(16, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)         
        return x

In [48]:
model = WineQualityModel(input_dim=X_train.shape[1])

In [49]:
criterion = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=0.001)


In [50]:
epochs = 100

for epoch in range(epochs):
    model.train()
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        outputs = model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        test_loss = 0
        for batch_X, batch_y in test_loader:
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            test_loss += loss.item()

    print(f'Epoch {epoch+1}/{epochs}, Test Loss: {test_loss/len(test_loader)}')

Epoch 1/100, Test Loss: 0.7441549375653267
Epoch 2/100, Test Loss: 0.6119122765958309
Epoch 3/100, Test Loss: 0.5711271837353706
Epoch 4/100, Test Loss: 0.545578483492136
Epoch 5/100, Test Loss: 0.5321297273039818
Epoch 6/100, Test Loss: 0.5368079133331776
Epoch 7/100, Test Loss: 0.5211609378457069
Epoch 8/100, Test Loss: 0.5211523473262787
Epoch 9/100, Test Loss: 0.5258920416235924
Epoch 10/100, Test Loss: 0.5185454562306404
Epoch 11/100, Test Loss: 0.508695624768734
Epoch 12/100, Test Loss: 0.5093594826757908
Epoch 13/100, Test Loss: 0.5099796392023563
Epoch 14/100, Test Loss: 0.4996022507548332
Epoch 15/100, Test Loss: 0.5008677467703819
Epoch 16/100, Test Loss: 0.49329874478280544
Epoch 17/100, Test Loss: 0.5050933063030243
Epoch 18/100, Test Loss: 0.48624108359217644
Epoch 19/100, Test Loss: 0.503980927169323
Epoch 20/100, Test Loss: 0.4896370694041252
Epoch 21/100, Test Loss: 0.49608289077878
Epoch 22/100, Test Loss: 0.48671145737171173
Epoch 23/100, Test Loss: 0.4976650811731815